# 02 — Global Symbolic Regression

Load the frozen CO₂ splits, train one global symbolic correction, and select
its equation using natural and high-density distribution-shift validation.
The ideal-gas baseline and reproducible summary tables are generated by
src/validate_results.py.


In [ ]:
!pip install -q pysr scikit-learn

In [ ]:
from pathlib import Path
import json
import pickle
import time

import numpy as np
import pandas as pd


def find_project_root():
    current = Path.cwd()
    candidates = [current, current.parent]

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
        if (candidate / "notebooks").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Processed data directory not found: {DATA_DIR}. "
        "Run 01_CO2_Data_Generation first."
    )

TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

train_df = pd.read_csv(DATA_DIR / "co2_train.csv")
validation_df = pd.read_csv(DATA_DIR / "co2_validation.csv")
test_df = pd.read_csv(DATA_DIR / "co2_test.csv")
stress_validation_df = pd.read_csv(
    DATA_DIR / "co2_stress_validation.csv"
)
stress_test_df = pd.read_csv(DATA_DIR / "co2_stress_test.csv")

FEATURE_COLUMNS = ["T_reduced", "rho_reduced"]
TARGET_COLUMN = "delta_Z"


In [ ]:
from pysr import PySRRegressor
import numpy as np
import pandas as pd
import time

FEATURE_COLUMNS = [
    "T_reduced",
    "rho_reduced"
]

TARGET_COLUMN = "delta_Z"

X_train = train_df[
    FEATURE_COLUMNS
].to_numpy()

y_train = train_df[
    TARGET_COLUMN
].to_numpy()

X_validation = validation_df[
    FEATURE_COLUMNS
].to_numpy()

y_validation = validation_df[
    TARGET_COLUMN
].to_numpy()

X_stress_validation = stress_validation_df[
    FEATURE_COLUMNS
].to_numpy()

y_stress_validation = stress_validation_df[
    TARGET_COLUMN
].to_numpy()

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print(
    "Stress validation shape:",
    X_stress_validation.shape
)

In [ ]:
global_model = PySRRegressor(
    niterations=100,
    populations=20,
    population_size=50,

    binary_operators=[
        "+",
        "-",
        "*",
        "/"
    ],

    unary_operators=[],

    maxsize=20,
    maxdepth=10,

    model_selection="best",

    elementwise_loss=(
        "loss(prediction, target) = "
        "(prediction - target)^2"
    ),

    parsimony=0.001,

    random_state=42,
    deterministic=True,
    parallelism="serial",

    verbosity=1
)

global_model

In [ ]:
start_time = time.time()

global_model.fit(
    X_train,
    y_train,
    variable_names=[
        "T_r",
        "rho_r"
    ]
)

training_time = time.time() - start_time

print(
    f"Training completed in "
    f"{training_time:.2f} seconds"
)

In [ ]:
equations = global_model.equations_.copy()

display(
    equations[
        [
            "complexity",
            "loss",
            "score",
            "equation"
        ]
    ]
)

In [ ]:
def evaluate_model(
    model,
    dataset,
    dataset_name,
    equation_index=None
):
    X = dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    true_delta_z = dataset[
        "delta_Z"
    ].to_numpy()

    if equation_index is None:
        predicted_delta_z = model.predict(X)
    else:
        predicted_delta_z = model.predict(
            X,
            index=equation_index
        )

    true_pressure = dataset[
        "p_real_Pa"
    ].to_numpy()

    ideal_pressure = dataset[
        "p_ideal_Pa"
    ].to_numpy()

    predicted_pressure = (
        ideal_pressure
        * (1.0 + predicted_delta_z)
    )

    delta_error = (
        predicted_delta_z - true_delta_z
    )

    pressure_relative_error = (
        np.abs(
            predicted_pressure - true_pressure
        )
        / np.abs(true_pressure)
        * 100.0
    )

    return {
        "dataset": dataset_name,

        "delta_Z_RMSE": np.sqrt(
            np.mean(delta_error ** 2)
        ),

        "delta_Z_MAE": np.mean(
            np.abs(delta_error)
        ),

        "pressure_MAPE_percent": np.mean(
            pressure_relative_error
        ),

        "pressure_max_error_percent": np.max(
            pressure_relative_error
        )
    }


In [ ]:
global_model_results = pd.DataFrame([
    evaluate_model(
        global_model,
        validation_df,
        "natural_validation"
    ),

    evaluate_model(
        global_model,
        stress_validation_df,
        "stress_validation"
    )
])

global_model_results


In [ ]:
def evaluate_by_regime(
    model,
    dataset,
    dataset_name
):
    results = []

    for regime_name, regime_df in dataset.groupby(
        "regime"
    ):
        result = evaluate_model(
            model,
            regime_df,
            dataset_name
        )

        result["regime"] = regime_name
        result["count"] = len(regime_df)

        results.append(result)

    return pd.DataFrame(results)


global_regime_results = pd.concat([
    evaluate_by_regime(
        global_model,
        validation_df,
        "natural_validation"
    ),

    evaluate_by_regime(
        global_model,
        stress_validation_df,
        "stress_validation"
    )
], ignore_index=True)

global_regime_results[
    [
        "dataset",
        "regime",
        "count",
        "delta_Z_RMSE",
        "delta_Z_MAE",
        "pressure_MAPE_percent"
    ]
]

In [ ]:
equations.to_csv(
    TABLE_DIR / "global_symbolic_equations.csv",
    index=False
)

global_model_results.to_csv(
    TABLE_DIR / "global_model_results.csv",
    index=False
)

global_regime_results.to_csv(
    TABLE_DIR / "global_regime_results.csv",
    index=False
)

with open(
    MODEL_DIR / "global_symbolic_model.pkl",
    "wb"
) as file:
    pickle.dump(global_model, file)

print("Global model and results saved to:", RESULTS_DIR)


In [ ]:
print(global_model.sympy())

## Equation selection

The test sets are not used for equation selection.

In [ ]:
def select_equation_by_validation(
    model,
    natural_dataset,
    stress_dataset,
    max_complexity=15
):
    rows = []

    natural_X = natural_dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    natural_y = natural_dataset[
        TARGET_COLUMN
    ].to_numpy()

    stress_X = stress_dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    stress_y = stress_dataset[
        TARGET_COLUMN
    ].to_numpy()

    for equation_index, equation_row in (
        model.equations_.iterrows()
    ):
        complexity = equation_row["complexity"]

        if complexity > max_complexity:
            continue

        natural_prediction = model.predict(
            natural_X,
            index=equation_index
        )

        stress_prediction = model.predict(
            stress_X,
            index=equation_index
        )

        natural_rmse = np.sqrt(
            np.mean(
                (
                    natural_prediction
                    - natural_y
                ) ** 2
            )
        )

        stress_rmse = np.sqrt(
            np.mean(
                (
                    stress_prediction
                    - stress_y
                ) ** 2
            )
        )

        # Natural distribution and stress region
        # receive equal importance.
        selection_score = (
            0.5 * natural_rmse
            + 0.5 * stress_rmse
        )

        rows.append({
            "equation_index": equation_index,
            "complexity": complexity,
            "natural_RMSE": natural_rmse,
            "stress_RMSE": stress_rmse,
            "selection_score": selection_score,
            "equation": equation_row["equation"]
        })

    result = pd.DataFrame(rows).sort_values(
        "selection_score"
    ).reset_index(drop=True)

    best_index = int(
        result.iloc[0]["equation_index"]
    )

    return best_index, result

In [ ]:
selected_global_index, global_selection_table = (
    select_equation_by_validation(
        global_model,
        validation_df,
        stress_validation_df
    )
)

print(
    "Selected global index:",
    selected_global_index
)

print(
    "Selected global equation:"
)

print(
    global_model.sympy(
        index=selected_global_index
    )
)

global_selection_table[
    [
        "equation_index",
        "complexity",
        "natural_RMSE",
        "stress_RMSE",
        "selection_score",
        "equation"
    ]
]

In [ ]:
global_validation_results = pd.DataFrame([
    evaluate_model(
        global_model,
        validation_df,
        "natural_validation",
        selected_global_index
    ),
    evaluate_model(
        global_model,
        stress_validation_df,
        "stress_validation",
        selected_global_index
    )
])

global_validation_results.to_csv(
    TABLE_DIR / "global_validation_results.csv",
    index=False
)

global_selection_table.to_csv(
    TABLE_DIR / "global_selection_table.csv",
    index=False
)

with open(MODEL_DIR / "global_selection.json", "w") as file:
    json.dump(
        {"selected_global_index": int(selected_global_index)},
        file,
        indent=2
    )

print("Selected global index saved:", selected_global_index)
